# Question 8  
In this exercise, we will generate simulated data, and will then use this data to perform forward and backward stepwise selection.

### (a)    
Create a random number generator and use its normal() method to generate a predictor $X$ of length $n = 100$, as well as a noise vector $\epsilon$ of length $n = 100$.

In [161]:
import numpy as np
n = 100
X = np.random.normal(0,1,n)
epsilon = np.random.normal(0,1,n) 

### (b)  
Generate a response vector $Y$ of length $n = 100$ according to the model$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \epsilon,$$where $\beta_0$, $\beta_1$, $\beta_2$, and $\beta_3$ are constants of your choice.

In [162]:
# set beta randomly
beta0 = 3
beta1 = 0.5
beta2 = -2
beta3 = 1

Y = beta0 + beta1*X + beta2*(X**2) + beta3*(X**3) + epsilon

### (c)  
Use forward stepwise selection in order to select a model containing the predictors $X, X^2, \dots, X^{10}$. What is the model obtained according to $C_p$? $$C_p = \frac{RSS_d}{\hat{\sigma}^2} - n + 2d$$ Report the coefficients of the model obtained.

In [163]:
import pandas as pd

data = pd.DataFrame({'Y':Y})
for i in range(1,11):
    data[f'X{i}'] = X**i
predictors = [f'X{i}' for i in range(1,11)]
print(data.head(5))


          Y        X1        X2        X3        X4        X5        X6  \
0  2.269610 -0.906568  0.821866 -0.745078  0.675464 -0.612354  0.555141   
1  1.443808  1.212777  1.470827  1.783785  2.163332  2.623639  3.181887   
2  1.629142 -0.128617  0.016542 -0.002128  0.000274 -0.000035  0.000005   
3  2.863687  1.086430  1.180331  1.282348  1.393181  1.513595  1.644415   
4  1.772633  0.947630  0.898002  0.850973  0.806407  0.764175  0.724155   

             X7            X8            X9           X10  
0 -5.032734e-01  4.562517e-01 -4.136234e-01  3.749779e-01  
1  3.858918e+00  4.680006e+00  5.675801e+00  6.883479e+00  
2 -5.822199e-07  7.488334e-08 -9.631266e-09  1.238744e-09  
3  1.786543e+00  1.940954e+00  2.108712e+00  2.290969e+00  
4  6.862307e-01  6.502925e-01  6.162364e-01  5.839638e-01  


In [164]:
def nCp(sigma2, estimator, X, Y):
    n, p_plus_one = X.shape
    p = p_plus_one - 1
    
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat)**2)

    return -(RSS + 2 * p * sigma2) / n

In [165]:
import numpy as np
import pandas as pd
from statsmodels.api import OLS
from ISLP.models import ModelSpec as MS
from functools import partial
from ISLP.models import Stepwise

Y = data['Y']
design = MS(predictors).fit(data)

X_all = design.transform(data)

full_model_fit = OLS(Y, X_all).fit()
sigma2 = full_model_fit.scale
neg_Cp = partial(nCp, sigma2)

In [166]:
import numpy as np
import pandas as pd
from statsmodels.api import OLS
from functools import partial
from ISLP.models import ModelSpec as MS, \
                        Stepwise, \
                        sklearn_selected

strategy = Stepwise.first_peak(design,
                               max_terms=len(design.terms))

best_model = sklearn_selected(strategy, X_all, Y, scoring=neg_Cp)

In [167]:
selector = sklearn_selected(OLS,
                            strategy,
                            scoring=neg_Cp)
selector.fit(X_all, Y) 

print("Selected variables (Forward Cp selection):")
print(selector.selected_state_)

Selected variables (Forward Cp selection):
('X2', 'X3', 'X4', 'X5', 'X7', 'X8')


### (d)  
Repeat (c), using backwards stepwise selection. How does your answer compare to the results in (c)?

In [168]:
from statsmodels.api import OLS, add_constant

def Cp(model, sigma2):
    RSS = np.sum(model.resid**2)
    p = len(model.params) - 1
    n = len(model.resid)
    return (RSS + 2*p*sigma2) / n

X_full = add_constant(data[predictors])
full_model = OLS(data['Y'], X_full).fit()
sigma2 = full_model.scale

current_predictors = predictors.copy()

while True:
    best_Cp = Cp(OLS(data['Y'], add_constant(data[current_predictors])).fit(), sigma2)
    worst_to_drop = None
    improved = False

    for var in current_predictors:
        trial = [v for v in current_predictors if v != var]
        model = OLS(data['Y'], add_constant(data[trial])).fit()
        cp_val = Cp(model, sigma2)
        
        if cp_val < best_Cp:
            best_Cp = cp_val
            worst_to_drop = var
            improved = True

    if improved:
        current_predictors.remove(worst_to_drop)
    else:
        break

print("Selected variables (Backward Cp selection):")
print(current_predictors)

Selected variables (Backward Cp selection):
['X2', 'X3', 'X4', 'X5', 'X7', 'X8']


**comparison:**  
With the current beta settings, forward selection captures the true predictors, while backward selection keeps some non-informative variables.

### (e)  
Now fit a lasso model to the simulated data, again using $X, X^2, \dots, X^{10}$ as predictors. Use cross-validation to select the optimal value of $\lambda$. Create plots of the cross-validation error as a function of $\lambda$. Report the resulting coefficient estimates, and discuss the results obtained.

In [169]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[predictors])

lasso = LassoCV(cv=10, random_state=42)
lasso.fit(X_scaled, data['Y'])

coef = pd.Series(lasso.coef_, index=predictors)
selected = coef[coef != 0].index.tolist()

print("LASSO selected variables:")
print(selected)

#print("\nCoefficients:")
#print(coef)

LASSO selected variables:
['X1', 'X2', 'X3', 'X7', 'X8', 'X10']


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.656e-01, tolerance: 2.397e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.772e-01, tolerance: 2.397e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check 

### (f)  
Now generate a response vector $Y$ according to the model$$Y = \beta_0 + \beta_7 X^7 + \epsilon,$$and perform forward stepwise selection and the lasso. Discuss the results obtained.

In [170]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

n = 100
X = np.random.normal(0,1,n)
epsilon = np.random.normal(0,1,n)

beta0 = -5
beta7 = 0.3

Y = beta0 + beta2*(X**7) + epsilon

data = pd.DataFrame({'Y':Y})
for i in range(1,11):
    data[f'X{i}'] = X**i

predictors = [f'X{i}' for i in range(1,11)]


In [171]:
# forward

def nCp(sigma2, estimator, X, Y):
    n, p_plus_one = X.shape
    p = p_plus_one - 1
    
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat)**2)

    return -(RSS + 2 * p * sigma2) / n

import numpy as np
import pandas as pd
from statsmodels.api import OLS
from ISLP.models import ModelSpec as MS
from functools import partial
from ISLP.models import Stepwise

Y = data['Y']
design = MS(predictors).fit(data)

X_all = design.transform(data)

full_model_fit = OLS(Y, X_all).fit()
sigma2 = full_model_fit.scale
neg_Cp = partial(nCp, sigma2)

import numpy as np
import pandas as pd
from statsmodels.api import OLS
from functools import partial
from ISLP.models import ModelSpec as MS, \
                        Stepwise, \
                        sklearn_selected

strategy = Stepwise.first_peak(design,
                               max_terms=len(design.terms))

best_model = sklearn_selected(strategy, X_all, Y, scoring=neg_Cp)

selector = sklearn_selected(OLS,
                            strategy,
                            scoring=neg_Cp)
selector.fit(X_all, Y) 

print("Selected variables (Forward Cp selection):")
print(selector.selected_state_)

Selected variables (Forward Cp selection):
('X7',)


In [172]:
# Lasso 

scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[predictors])

lasso = LassoCV(cv=10, random_state=42)
lasso.fit(X_scaled, data['Y'])

coef = pd.Series(lasso.coef_, index=predictors)
selected = coef[coef != 0].index.tolist()

print("LASSO selected variables:")
print(selected)

#print("\nCoefficients:")
#print(coef)

LASSO selected variables:
['X1', 'X5', 'X7', 'X9']


**comparison:**  
With the current beta settings, forward selection still captures the true predictors, while lasso keeps some non-informative variables.